In [ ]:
import os
import os,zipfile
import torch
from PIL import Image
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
import pandas as pd

folder="/content/drive/MyDrive/SNN_XAI_PROJECT"
os.makedirs(folder,exist_ok=True)
print("Project folder created:",os.path.exists(folder))
print("Folder location:",folder)

Project folder created: True
Folder location: /content/drive/MyDrive/SNN_XAI_PROJECT


In [ ]:
#
# folder="/content/drive/MyDrive/SNN_XAI_PROJECT"
# zip_path=f"{folder}/MILK10k_Training_Input.zip"
# print("ZIP Found:",os.path.isfile(zip_path))
# if os.path.isfile(zip_path):
#     print("ZIP Size (GB):",round(os.path.getsize(zip_path)/1024**3,2))
#     with zipfile.ZipFile(zip_path,"r") as z:
#         print("Total Files:",len(z.namelist()))
#         print("First 5 Entries:",z.namelist()[:5])

ZIP Found: True
ZIP Size (GB): 0.31
Total Files: 15724
First 5 Entries: ['MILK10k_Training_Input/', 'MILK10k_Training_Input/IL_0000652/', 'MILK10k_Training_Input/IL_0000652/ISIC_4671410.jpg', 'MILK10k_Training_Input/IL_0000652/ISIC_8149219.jpg', 'MILK10k_Training_Input/IL_0003176/']


In [ ]:
# 3
folder="/content/drive/MyDrive/SNN_XAI_PROJECT"
zip_path=f"{folder}/MILK10k_Training_Input.zip"
extract_path=f"{folder}/images"
os.makedirs(extract_path,exist_ok=True)
with zipfile.ZipFile(zip_path,"r") as z:
    z.extractall(extract_path)
image_folder=f"{extract_path}/MILK10k_Training_Input"
print("Extraction Complete!")
print("Dataset Folder:",image_folder)
print("Total Lesion Folders:",len([x for x in os.listdir(image_folder) if os.path.isdir(os.path.join(image_folder,x))]))

Extraction Complete!
Dataset Folder: /content/drive/MyDrive/SNN_XAI_PROJECT/images/MILK10k_Training_Input
Total Lesion Folders: 5241


In [ ]:
# 4
folder="/content/drive/MyDrive/SNN_XAI_PROJECT"
for root,dirs,files in os.walk(folder):
    for file in files:
        if file.lower().endswith(".csv"):
            print(os.path.join(root,file))

/content/drive/MyDrive/SNN_XAI_PROJECT/MILK10k_Training_Supplement.csv
/content/drive/MyDrive/SNN_XAI_PROJECT/MILK10k_Training_Metadata.csv
/content/drive/MyDrive/SNN_XAI_PROJECT/MILK10k_Training_GroundTruth.csv
/content/drive/MyDrive/SNN_XAI_PROJECT/train.csv
/content/drive/MyDrive/SNN_XAI_PROJECT/val.csv
/content/drive/MyDrive/SNN_XAI_PROJECT/test.csv


In [ ]:
# 5
folder="/content/drive/MyDrive/SNN_XAI_PROJECT"
gt=pd.read_csv(f"{folder}/MILK10k_Training_GroundTruth.csv")
meta=pd.read_csv(f"{folder}/MILK10k_Training_Metadata.csv")
supp=pd.read_csv(f"{folder}/MILK10k_Training_Supplement.csv")
print("Ground Truth:",gt.shape)
print("Ground Truth Columns:",gt.columns.tolist())
print("\nMetadata:",meta.shape)
print("Metadata Columns:",meta.columns.tolist())
print("\nSupplement:",supp.shape)
print("Supplement Columns:",supp.columns.tolist())
print("\nGround Truth Preview:")
print(gt.head(3).to_string(index=False))

Ground Truth: (5240, 12)
Ground Truth Columns: ['lesion_id', 'AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']

Metadata: (10480, 17)
Metadata Columns: ['lesion_id', 'image_type', 'isic_id', 'attribution', 'copyright_license', 'image_manipulation', 'age_approx', 'sex', 'skin_tone_class', 'site', 'MONET_ulceration_crust', 'MONET_hair', 'MONET_vasculature_vessels', 'MONET_erythema', 'MONET_pigmented', 'MONET_gel_water_drop_fluid_dermoscopy_liquid', 'MONET_skin_markings_pen_ink_purple_pen']

Supplement: (10480, 4)
Supplement Columns: ['isic_id', 'diagnosis_full', 'diagnosis_confirm_type', 'invasion_thickness_interval']

Ground Truth Preview:
 lesion_id  AKIEC  BCC  BEN_OTH  BKL  DF  INF  MAL_OTH  MEL  NV  SCCKA  VASC
IL_0000652    0.0  1.0      0.0  0.0 0.0  0.0      0.0  0.0 0.0    0.0   0.0
IL_0003176    0.0  1.0      0.0  0.0 0.0  0.0      0.0  0.0 0.0    0.0   0.0
IL_0004688    0.0  1.0      0.0  0.0 0.0  0.0      0.0  0.0 0.0    0.0   0.0


In [ ]:
# 6
classes=["AKIEC","BCC","BEN_OTH","BKL","DF","INF","MAL_OTH","MEL","NV","SCCKA","VASC"]
print("Class Distribution:")
print(gt[classes].sum().astype(int).to_string())
print("\nImage Types:")
print(meta["image_type"].value_counts(dropna=False).to_string())
print("\nLabels per Lesion:")
print(gt[classes].sum(axis=1).value_counts().to_string())
print("\nImages per Lesion:")
print(meta.groupby("lesion_id").size().value_counts().sort_index().to_string())
print("\nGround Truth Lesions Missing from Metadata:",len(set(gt["lesion_id"])-set(meta["lesion_id"])))
print("Metadata Lesions Missing from Ground Truth:",len(set(meta["lesion_id"])-set(gt["lesion_id"])))

Class Distribution:
AKIEC       303
BCC        2522
BEN_OTH      44
BKL         544
DF           52
INF          50
MAL_OTH       9
MEL         450
NV          746
SCCKA       473
VASC         47

Image Types:
image_type
clinical: close-up    5240
dermoscopic           5240

Labels per Lesion:
1.0    5240

Images per Lesion:
2    5240

Ground Truth Lesions Missing from Metadata: 0
Metadata Lesions Missing from Ground Truth: 0


In [ ]:
# 7
classes=["AKIEC","BCC","BEN_OTH","BKL","DF","INF","MAL_OTH","MEL","NV","SCCKA","VASC"]
labels=gt[["lesion_id"]+classes].copy()
labels["label"]=labels[classes].idxmax(axis=1)
labels=labels[["lesion_id","label"]]
pairs=meta.pivot(index="lesion_id",columns="image_type",values="isic_id").reset_index()
pairs=pairs.rename(columns={"dermoscopic":"dermoscopic_id","clinical: close-up":"closeup_id"})
dataset=pairs.merge(labels,on="lesion_id",how="inner",validate="one_to_one")
assert len(dataset)==5240
assert dataset[["dermoscopic_id","closeup_id","label"]].notna().all().all()
print("Total Paired Lesions:",len(dataset))
print("\nDataset Preview:")
print(dataset.head().to_string(index=False))

Total Paired Lesions: 5240

Dataset Preview:
 lesion_id   closeup_id dermoscopic_id label
IL_0000652 ISIC_8149219   ISIC_4671410   BCC
IL_0003176 ISIC_3904045   ISIC_5371928   BCC
IL_0004688 ISIC_0791494   ISIC_3624913   BCC
IL_0005081 ISIC_5667730   ISIC_5186409 SCCKA
IL_0006177 ISIC_8803389   ISIC_1048297   BCC


In [ ]:
# 8
from sklearn.model_selection import train_test_split
train_df,temp_df=train_test_split(dataset,test_size=0.30,stratify=dataset["label"],random_state=42)
val_df,test_df=train_test_split(temp_df,test_size=0.50,stratify=temp_df["label"],random_state=42)
train_df.to_csv(f"{folder}/train.csv",index=False)
val_df.to_csv(f"{folder}/val.csv",index=False)
test_df.to_csv(f"{folder}/test.csv",index=False)
print("Training:",len(train_df))
print("Validation:",len(val_df))
print("Testing:",len(test_df))
print("\nTraining Class Distribution:")
print(train_df["label"].value_counts().reindex(classes,fill_value=0).to_string())

Training: 3668
Validation: 786
Testing: 786

Training Class Distribution:
label
AKIEC       212
BCC        1765
BEN_OTH      31
BKL         381
DF           37
INF          35
MAL_OTH       6
MEL         315
NV          522
SCCKA       331
VASC         33


In [ ]:
# 9 resizing batch size and timesteps
image_folder=f"{folder}/images/MILK10k_Training_Input"
classes=["AKIEC","BCC","BEN_OTH","BKL","DF","INF","MAL_OTH","MEL","NV","SCCKA","VASC"]
transform=transforms.Compose([transforms.Resize((96,128)),transforms.Pad((0,16,0,16)),transforms.ToTensor()])
class MILK10kDataset(Dataset):
    def __init__(self,csv_file):
        self.data=pd.read_csv(csv_file)
    def __len__(self):
        return len(self.data)
    def __getitem__(self,idx):
        row=self.data.iloc[idx]
        def load_image(image_id):
            path=f"{image_folder}/{row['lesion_id']}/{image_id}.jpg"
            with Image.open(path) as img:
                return transform(img.convert("RGB"))
        derm=load_image(row["dermoscopic_id"])
        closeup=load_image(row["closeup_id"])
        label=classes.index(row["label"])
        return derm,closeup,label
train_loader=DataLoader(MILK10kDataset(f"{folder}/train.csv"),batch_size=2,shuffle=True,num_workers=0)
val_loader=DataLoader(MILK10kDataset(f"{folder}/val.csv"),batch_size=2,shuffle=False,num_workers=0)
test_loader=DataLoader(MILK10kDataset(f"{folder}/test.csv"),batch_size=2,shuffle=False,num_workers=0)
print("Training Batches:",len(train_loader))
print("Validation Batches:",len(val_loader))
print("Testing Batches:",len(test_loader))

Training Batches: 1834
Validation Batches: 393
Testing Batches: 393


In [ ]:
# 10
derm,closeup,labels=next(iter(train_loader))
print("Dermoscopic Shape:",derm.shape)
print("Close-up Shape:",closeup.shape)
print("Labels Shape:",labels.shape)
print("Pixel Range:",derm.min().item(),derm.max().item())

Dermoscopic Shape: torch.Size([2, 3, 128, 128])
Close-up Shape: torch.Size([2, 3, 128, 128])
Labels Shape: torch.Size([2])
Pixel Range: 0.0 0.8274509906768799


In [ ]:
%pip install -q snntorch

In [ ]:
# 11 architecture of spiking vgg
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
# if
# class SpikingConvBlock(nn.Module):
#     def __init__(self,in_channels,out_channels):
#         super().__init__()
#         self.conv1=nn.Conv2d(in_channels,out_channels,3,padding=1)
#         self.bn1=nn.BatchNorm2d(out_channels)
#         self.if1=snn.Leaky(beta=1.0,spike_grad=surrogate.fast_sigmoid())
#         self.drop1=nn.Dropout2d(0.2)
#         self.conv2=nn.Conv2d(out_channels,out_channels,3,padding=1)
#         self.bn2=nn.BatchNorm2d(out_channels)
#         self.if2=snn.Leaky(beta=1.0,spike_grad=surrogate.fast_sigmoid())
#         self.drop2=nn.Dropout2d(0.2)
#     def forward(self,x,mem1,mem2):
#         spk1,mem1=self.if1(self.bn1(self.conv1(x)),mem1)
#         x=self.drop1(spk1)
#         spk2,mem2=self.if2(self.bn2(self.conv2(x)),mem2)
#         x=self.drop2(spk2)
#         return x,mem1,mem2
# lif
class SpikingConvBlock(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels,out_channels,3,padding=1)
        self.bn1=nn.BatchNorm2d(out_channels)
        self.if1=snn.Leaky(beta=0.9,spike_grad=surrogate.fast_sigmoid())
        self.drop1=nn.Dropout2d(0.2)
        self.conv2=nn.Conv2d(out_channels,out_channels,3,padding=1)
        self.bn2=nn.BatchNorm2d(out_channels)
        self.if2=snn.Leaky(beta=0.9,spike_grad=surrogate.fast_sigmoid())
        self.drop2=nn.Dropout2d(0.2)
    def forward(self,x,mem1,mem2):
        spk1,mem1=self.if1(self.bn1(self.conv1(x)),mem1)
        x=self.drop1(spk1)
        spk2,mem2=self.if2(self.bn2(self.conv2(x)),mem2)
        x=self.drop2(spk2)
        return x,mem1,mem2

In [ ]:
# 12 (inside convolution)
class SpikingVGG13(nn.Module):
    def __init__(self):
        super().__init__()
        channels=[3,64,128,256,512,512]
        self.blocks=nn.ModuleList([SpikingConvBlock(channels[i],channels[i+1]) for i in range(5)])
        self.pool=nn.AvgPool2d(2)
    def forward(self,x,states):
        new_states=[]
        for i,block in enumerate(self.blocks):
            x,mem1,mem2=block(x,*states[i])
            new_states.append((mem1,mem2))
            x=self.pool(x)
        return x,new_states

In [ ]:
# # 13( mein ek branch dermoscopic images process horahi aur doosri clinical close-up images. Dono branches ke features concatenate ho rahe aur final classification 11 classes mein hogi.)
# class DualBranchSpikingVGG13(nn.Module):
#     def __init__(self,num_classes=11,num_steps=30):
#         super().__init__()
#         self.num_steps=num_steps
#         self.derm_branch=SpikingVGG13()
#         self.closeup_branch=SpikingVGG13()
#         self.fc1=nn.Linear(2*512*4*4,1024)
#         self.if_fc1=snn.Leaky(beta=1.0,spike_grad=surrogate.fast_sigmoid())
#         self.drop1=nn.Dropout(0.5)
#         self.fc2=nn.Linear(1024,num_classes)
#         self.if_fc2=snn.Leaky(beta=1.0,spike_grad=surrogate.fast_sigmoid())
#     def init_states(self,x):
#         channels=[64,128,256,512,512]
#         sizes=[128,64,32,16,8]
#         return [(x.new_zeros(x.size(0),ch,s,s),x.new_zeros(x.size(0),ch,s,s)) for ch,s in zip(channels,sizes)]
#     def forward(self,derm,closeup):
#         derm_states=self.init_states(derm)
#         closeup_states=self.init_states(closeup)
#         mem_fc1=derm.new_zeros(derm.size(0),1024)
#         mem_fc2=derm.new_zeros(derm.size(0),self.fc2.out_features)
#         spike_counts=derm.new_zeros(derm.size(0),self.fc2.out_features)
#         for t in range(self.num_steps):
#             derm_spikes=torch.bernoulli(derm)
#             closeup_spikes=torch.bernoulli(closeup)
#             derm_features,derm_states=self.derm_branch(derm_spikes,derm_states)
#             closeup_features,closeup_states=self.closeup_branch(closeup_spikes,closeup_states)
#             fused=torch.cat((torch.flatten(derm_features,1),torch.flatten(closeup_features,1)),dim=1)
#             spk_fc1,mem_fc1=self.if_fc1(self.fc1(fused),mem_fc1)
#             spk_fc1=self.drop1(spk_fc1)
#             spk_fc2,mem_fc2=self.if_fc2(self.fc2(spk_fc1),mem_fc2)
#             spike_counts=spike_counts+spk_fc2
#         return spike_counts
class DualBranchSpikingVGG13(nn.Module):
    def __init__(self,num_classes=11,num_steps=30):
        super().__init__()
        self.num_steps=num_steps
        self.derm_branch=SpikingVGG13()
        self.closeup_branch=SpikingVGG13()
        self.fc1=nn.Linear(2*512*4*4,1024)
        self.if_fc1=snn.Leaky(beta=0.9,spike_grad=surrogate.fast_sigmoid())
        self.drop1=nn.Dropout(0.5)
        self.fc2=nn.Linear(1024,num_classes)
        self.if_fc2=snn.Leaky(beta=0.9,spike_grad=surrogate.fast_sigmoid())
    def init_states(self,x):
        channels=[64,128,256,512,512]
        sizes=[128,64,32,16,8]
        return [(x.new_zeros(x.size(0),ch,s,s),x.new_zeros(x.size(0),ch,s,s)) for ch,s in zip(channels,sizes)]
    def forward(self,derm,closeup):
        derm_states=self.init_states(derm)
        closeup_states=self.init_states(closeup)
        mem_fc1=derm.new_zeros(derm.size(0),1024)
        mem_fc2=derm.new_zeros(derm.size(0),self.fc2.out_features)
        spike_counts=derm.new_zeros(derm.size(0),self.fc2.out_features)
        for t in range(self.num_steps):
            derm_spikes=torch.bernoulli(derm)
            closeup_spikes=torch.bernoulli(closeup)
            derm_features,derm_states=self.derm_branch(derm_spikes,derm_states)
            closeup_features,closeup_states=self.closeup_branch(closeup_spikes,closeup_states)
            fused=torch.cat((torch.flatten(derm_features,1),torch.flatten(closeup_features,1)),dim=1)
            spk_fc1,mem_fc1=self.if_fc1(self.fc1(fused),mem_fc1)
            spk_fc1=self.drop1(spk_fc1)
            spk_fc2,mem_fc2=self.if_fc2(self.fc2(spk_fc1),mem_fc2)
            spike_counts=spike_counts+spk_fc2
        return spike_counts




In [ ]:
# # check kar rahe hain ke model GPU mein fit hota hai ya nahi, phir check karenge ke GPU us model ko train bhi kar sakta hai ya nahi
# import gc
# gc.collect()
# torch.cuda.empty_cache()
# device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model=DualBranchSpikingVGG13(num_classes=11,num_steps=30).to(device)
# total_params=sum(p.numel() for p in model.parameters())
# trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
# print("Device:",device)
# print("Total Parameters:",f"{total_params:,}")
# print("Trainable Parameters:",f"{trainable_params:,}")
# if torch.cuda.is_available():
#     print("GPU:",torch.cuda.get_device_name(0))
#     print("GPU Memory Allocated:",round(torch.cuda.memory_allocated()/1024**3,2),"GB")
#     print("GPU Memory Reserved:",round(torch.cuda.memory_reserved()/1024**3,2),"GB")

Device: cuda
Total Parameters: 35,611,275
Trainable Parameters: 35,611,275
GPU: NVIDIA A100-SXM4-40GB
GPU Memory Allocated: 0.13 GB
GPU Memory Reserved: 0.29 GB


In [ ]:
# 14
import torch.optim as optim
criterion=nn.CrossEntropyLoss()
optimizer=optim.SGD(model.parameters(),lr=0.003)
num_epochs=10
model.num_steps=30
print("Loss Function:",criterion)
print("Optimizer:",optimizer.__class__.__name__)

Loss Function: CrossEntropyLoss()
Optimizer: SGD


In [ ]:
# from tqdm.auto import tqdm
# checkpoint_dir=f"{folder}/checkpoints"
# os.makedirs(checkpoint_dir,exist_ok=True)
# best_val_acc=-1
# for epoch in range(num_epochs):
#     model.train()
#     train_loss=0
#     train_correct=0
#     for derm,closeup,labels in tqdm(train_loader,desc=f"Epoch {epoch+1}/{num_epochs}"):
#         derm,closeup,labels=derm.to(device),closeup.to(device),labels.to(device)
#         optimizer.zero_grad(set_to_none=True)
#         outputs=model(derm,closeup)
#         loss=criterion(outputs,labels)
#         loss.backward()
#         optimizer.step()
#         train_loss+=loss.item()*labels.size(0)
#         train_correct+=(outputs.argmax(dim=1)==labels).sum().item()
#     model.eval()
#     val_loss=0
#     val_correct=0
#     with torch.no_grad():
#         for derm,closeup,labels in val_loader:
#             derm,closeup,labels=derm.to(device),closeup.to(device),labels.to(device)
#             outputs=model(derm,closeup)
#             val_loss+=criterion(outputs,labels).item()*labels.size(0)
#             val_correct+=(outputs.argmax(dim=1)==labels).sum().item()
#     train_acc=100*train_correct/len(train_loader.dataset)
#     val_acc=100*val_correct/len(val_loader.dataset)
#     print(f"Epoch {epoch+1}: Train Loss={train_loss/len(train_loader.dataset):.4f}, Train Acc={train_acc:.2f}%, Val Loss={val_loss/len(val_loader.dataset):.4f}, Val Acc={val_acc:.2f}%",flush=True)
#     checkpoint={"epoch":epoch+1,"model_state_dict":model.state_dict(),"optimizer_state_dict":optimizer.state_dict(),"best_val_acc":max(best_val_acc,val_acc)}
#     torch.save(checkpoint,f"{checkpoint_dir}/latest_checkpoint.pth")
#     if val_acc>best_val_acc:
#         best_val_acc=val_acc
#         torch.save(checkpoint,f"{checkpoint_dir}/best_model.pth")
#     print("Checkpoint Saved!",flush=True)

Epoch 1/10:   0%|          | 0/459 [00:00<?, ?it/s]

Epoch 1: Train Loss=2.4413, Train Acc=5.92%, Val Loss=2.3979, Val Acc=5.85%
Checkpoint Saved!


Epoch 2/10:   0%|          | 0/459 [00:00<?, ?it/s]

Epoch 2: Train Loss=2.3979, Train Acc=5.78%, Val Loss=2.3979, Val Acc=5.85%
Checkpoint Saved!


Epoch 3/10:   0%|          | 0/459 [00:00<?, ?it/s]

Epoch 3: Train Loss=2.3979, Train Acc=5.78%, Val Loss=2.3979, Val Acc=5.85%
Checkpoint Saved!


... (4 outputs left out)

Epoch 6/10:   0%|          | 0/459 [00:00<?, ?it/s]

Epoch 6: Train Loss=2.3979, Train Acc=5.78%, Val Loss=2.3979, Val Acc=5.85%
Checkpoint Saved!


Epoch 7/10:   0%|          | 0/459 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# model.eval()
# derm,closeup,labels=next(iter(val_loader))
# with torch.no_grad():
#     outputs=model(derm.to(device),closeup.to(device))
# print("Output Spike Counts:",outputs[0].cpu().tolist())
# print("Total Spikes:",outputs.sum().item())
# print("Predicted Classes:",outputs.argmax(dim=1).cpu().tolist())       batch ki 8 classes ko AKIE predict kia

Output Spike Counts: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Total Spikes: 0.0
Predicted Classes: [0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
# Ye numbers ek batch ke 30 time steps mein total spike events hain.
# model.eval()
# activity={}
# hooks=[]
# def monitor(name):
#     def hook(module,inputs,output):
#         spikes=output[0] if isinstance(output,tuple) else output
#         activity[name]=activity.get(name,0)+spikes.detach().sum().item()
#     return hook
# for branch_name in ["derm_branch","closeup_branch"]:
#     branch=getattr(model,branch_name)
#     for i,block in enumerate(branch.blocks):
#         hooks.append(block.if1.register_forward_hook(monitor(f"{branch_name}_block{i+1}_IF1")))
#         hooks.append(block.if2.register_forward_hook(monitor(f"{branch_name}_block{i+1}_IF2")))
# hooks.append(model.if_fc1.register_forward_hook(monitor("FC1")))
# hooks.append(model.if_fc2.register_forward_hook(monitor("FC2")))
# with torch.no_grad():
#     outputs=model(derm.to(device),closeup.to(device))
# for h in hooks:
#     h.remove()
# for name,count in activity.items():
#     print(name,":",int(count))

derm_branch_block1_IF1 : 51017184
derm_branch_block1_IF2 : 56868027
derm_branch_block2_IF1 : 29942875
derm_branch_block2_IF2 : 27734659
derm_branch_block3_IF1 : 14920852
derm_branch_block3_IF2 : 14024861
derm_branch_block4_IF1 : 7866696
derm_branch_block4_IF2 : 7576444
derm_branch_block5_IF1 : 1994959
derm_branch_block5_IF2 : 1869155
closeup_branch_block1_IF1 : 53263442
closeup_branch_block1_IF2 : 54224998
closeup_branch_block2_IF1 : 29269469
closeup_branch_block2_IF2 : 28050987
closeup_branch_block3_IF1 : 15087913
closeup_branch_block3_IF2 : 14281929
closeup_branch_block4_IF1 : 7706490
closeup_branch_block4_IF2 : 7382835
closeup_branch_block5_IF1 : 1939854
closeup_branch_block5_IF2 : 1891081
FC1 : 39384
FC2 : 0


In [ ]:
# model.eval()
# fc2_mem=[]
# def monitor_fc2(module,inputs,output):
#     fc2_mem.append(output[1].detach().cpu())
# hook=model.if_fc2.register_forward_hook(monitor_fc2)
# with torch.no_grad():
#     outputs=model(derm.to(device),closeup.to(device))
# hook.remove()
# mem=torch.stack(fc2_mem)
# print("FC2 Maximum Membrane Potential:",mem.max().item())
# print("FC2 Minimum Membrane Potential:",mem.min().item())
# print("FC2 Mean Membrane Potential:",mem.mean().item())
# print("FC2 Threshold:",model.if_fc2.threshold)
# print("Total Output Spikes:",outputs.sum().item())

FC2 Maximum Membrane Potential: 0.027202052995562553
FC2 Minimum Membrane Potential: -257.8676452636719
FC2 Mean Membrane Potential: -37.1823616027832
FC2 Threshold: tensor(1., device='cuda:0')
Total Output Spikes: 0.0


In [ ]:
# # 2 training
# # LIF training: beta=0.9
# import os,torch
# from tqdm.auto import tqdm
# checkpoint_dir=f"{folder}/checkpoints_LIF_beta09"
# os.makedirs(checkpoint_dir,exist_ok=True)
# best_val_acc=-1
# for epoch in range(num_epochs):
#     model.train()
#     train_loss=0
#     train_correct=0
#     for derm,closeup,labels in tqdm(train_loader,desc=f"LIF Epoch {epoch+1}/{num_epochs}"):
#         derm,closeup,labels=derm.to(device),closeup.to(device),labels.to(device)
#         optimizer.zero_grad(set_to_none=True)
#         outputs=model(derm,closeup)
#         loss=criterion(outputs,labels)
#         loss.backward()
#         optimizer.step()
#         train_loss+=loss.item()*labels.size(0)
#         train_correct+=(outputs.argmax(dim=1)==labels).sum().item()
#     model.eval()
#     val_loss=0
#     val_correct=0
#     with torch.no_grad():
#         for derm,closeup,labels in val_loader:
#             derm,closeup,labels=derm.to(device),closeup.to(device),labels.to(device)
#             outputs=model(derm,closeup)
#             val_loss+=criterion(outputs,labels).item()*labels.size(0)
#             val_correct+=(outputs.argmax(dim=1)==labels).sum().item()
#     train_acc=100*train_correct/len(train_loader.dataset)
#     val_acc=100*val_correct/len(val_loader.dataset)
#     print(f"Epoch {epoch+1}: Train Loss={train_loss/len(train_loader.dataset):.4f}, Train Acc={train_acc:.2f}%, Val Loss={val_loss/len(val_loader.dataset):.4f}, Val Acc={val_acc:.2f}%",flush=True)
#     checkpoint={"epoch":epoch+1,"model_state_dict":model.state_dict(),"optimizer_state_dict":optimizer.state_dict(),"best_val_acc":max(best_val_acc,val_acc),"beta":0.9,"num_steps":model.num_steps}
#     torch.save(checkpoint,f"{checkpoint_dir}/latest_checkpoint.pth")
#     if val_acc>best_val_acc:
#         best_val_acc=val_acc
#         torch.save(checkpoint,f"{checkpoint_dir}/best_model.pth")
#     print("LIF Checkpoint Saved!",flush=True)


LIF Epoch 1/10:   0%|          | 0/1834 [00:00<?, ?it/s]

Epoch 1: Train Loss=2.4236, Train Acc=5.53%, Val Loss=2.3978, Val Acc=5.98%
LIF Checkpoint Saved!


LIF Epoch 2/10:   0%|          | 0/1834 [00:00<?, ?it/s]

Epoch 2: Train Loss=2.4218, Train Acc=5.89%, Val Loss=2.4005, Val Acc=5.85%
LIF Checkpoint Saved!


LIF Epoch 3/10:   0%|          | 0/1834 [00:00<?, ?it/s]

Epoch 3: Train Loss=2.4225, Train Acc=5.78%, Val Loss=2.4030, Val Acc=5.60%
LIF Checkpoint Saved!


LIF Epoch 4/10:   0%|          | 0/1834 [00:00<?, ?it/s]

Epoch 4: Train Loss=2.4252, Train Acc=5.43%, Val Loss=2.3996, Val Acc=5.85%
LIF Checkpoint Saved!


LIF Epoch 5/10:   0%|          | 0/1834 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:

# Check LIF output spike activity
model.eval()
derm,closeup,labels=next(iter(val_loader))
with torch.no_grad():
    outputs=model(derm.to(device),closeup.to(device))
print("Output Spike Counts:",outputs.cpu().tolist())
print("Total Output Spikes:",outputs.sum().item())
print("Predicted Classes:",outputs.argmax(dim=1).cpu().tolist())


Output Spike Counts: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]
Total Output Spikes: 0.0
Predicted Classes: [0, 0]


In [ ]:

# Diagnose FC2 input current and gradient flow
model.train()
derm,closeup,labels=next(iter(train_loader))
derm,closeup,labels=derm.to(device),closeup.to(device),labels.to(device)
currents=[]
def check_fc2(module,inputs,output):
    currents.append(output.detach())
hook=model.fc2.register_forward_hook(check_fc2)
optimizer.zero_grad(set_to_none=True)
outputs=model(derm,closeup)
loss=criterion(outputs,labels)
loss.backward()
hook.remove()
current=torch.stack(currents)
print("FC2 Input Current Min:",current.min().item())
print("FC2 Input Current Max:",current.max().item())
print("FC2 Input Current Mean:",current.mean().item())
print("FC2 Weight Gradient:",None if model.fc2.weight.grad is None else model.fc2.weight.grad.norm().item())
print("FC1 Weight Gradient:",None if model.fc1.weight.grad is None else model.fc1.weight.grad.norm().item())
print("Loss:",loss.item())
optimizer.zero_grad(set_to_none=True)


FC2 Input Current Min: -0.45440539717674255
FC2 Input Current Max: 0.42542919516563416
FC2 Input Current Mean: -0.006725363899022341
FC2 Weight Gradient: 4754.2490234375
FC1 Weight Gradient: 8246.4052734375
Loss: 2.397895336151123


In [ ]:

# Test gradient clipping on one training batch
model.train()
derm,closeup,labels=next(iter(train_loader))
derm,closeup,labels=derm.to(device),closeup.to(device),labels.to(device)
optimizer.zero_grad(set_to_none=True)
outputs=model(derm,closeup)
loss=criterion(outputs,labels)
loss.backward()
grad_before=torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
print("Loss before update:",loss.item())
print("Gradient norm before clipping:",grad_before.item())
optimizer.step()
optimizer.zero_grad(set_to_none=True)
model.eval()
with torch.no_grad():
    new_outputs=model(derm,closeup)
    new_loss=criterion(new_outputs,labels)
print("Loss after update:",new_loss.item())
print("Output spikes after update:",new_outputs.sum().item())


Loss before update: 2.397895336151123
Gradient norm before clipping: 9865.015625
Loss after update: 2.470468044281006
Output spikes after update: 1.0


In [ ]:

# Diagnostic: Can the LIF model learn one batch?
import copy
test_model=copy.deepcopy(model).to(device)
test_optimizer=torch.optim.SGD(test_model.parameters(),lr=0.0001)
derm,closeup,labels=next(iter(train_loader))
derm,closeup,labels=derm.to(device),closeup.to(device),labels.to(device)
for step in range(10):
    test_model.train()
    test_optimizer.zero_grad(set_to_none=True)
    outputs=test_model(derm,closeup)
    loss=criterion(outputs,labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(test_model.parameters(),1.0)
    test_optimizer.step()
    print(f"Step {step+1}: Loss={loss.item():.4f}, Spikes={outputs.sum().item()}")
del test_model,test_optimizer
torch.cuda.empty_cache()


Step 1: Loss=2.3979, Spikes=0.0
Step 2: Loss=2.3979, Spikes=0.0
Step 3: Loss=2.3979, Spikes=0.0
Step 4: Loss=2.3979, Spikes=0.0
Step 5: Loss=2.4705, Spikes=1.0
Step 6: Loss=2.4705, Spikes=1.0
Step 7: Loss=2.3979, Spikes=0.0
Step 8: Loss=2.4705, Spikes=1.0
Step 9: Loss=2.3979, Spikes=0.0
Step 10: Loss=2.4705, Spikes=1.0
